**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/08-implement-your-own-paper.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/project-1-brief) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Antigravity](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7H · Implement Your Own Paper: Template + Worked Example

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| worked example 30 min · your paper 1–2 weeks | Google Colab (CPU is enough, GPU is faster) or Google Antigravity / VS Code on your laptop | a complete paper implementation following the 7-step recipe of the lecture: first a worked example (Squeeze-and-Excitation Networks), then your own paper for Project 1 |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

## How to use this notebook
1. **Part A** walks through the whole recipe on a small, real CVPR paper. Read it and run it (≈ 30 min).
2. **File → Save a copy in Drive** (Colab) or copy the file (Antigravity), rename it `paper-<short-name>.ipynb`, delete Part A,
   and fill **Part B** for your own paper. One notebook per paper.
3. **Project 1** (see the [labs page](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-project-1)):
   Labs 7C–7F count as your first papers; add **at least one more** from the lecture's menu (Level 8).

### The 7-step recipe (Lecture 7, Level 2)
| Step | What | Output in this notebook |
|---|---|---|
| 1 | **Pass 1 (10 min):** title, abstract, figures, conclusion | Paper card |
| 2 | **Pass 2 (1 h):** method section, the architecture figure, the main table | Notes + shape table |
| 3 | **Map figure → modules**, equations → code | Module list, equation table |
| 4 | **Build bottom-up**, test every block (shapes, a value you can check by hand, parameter count) | `assert`s and checks |
| 5 | **Overfit one batch** | loss → ≈ 0 |
| 6 | **Small experiment** that tests the paper's main claim (with an ablation) | results table + plot |
| 7 | **Report**: what matched, what didn't, why; what you'd do with more compute | reproduction report |

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

---
# Part A · Worked example: Squeeze-and-Excitation Networks

## Step 1 · Paper card (pass 1, 10 minutes)

| | |
|---|---|
| **Title** | Squeeze-and-Excitation Networks |
| **Authors, venue, year** | J. Hu, L. Shen, G. Sun (+ S. Albanie, E. Wu in the journal version) · CVPR 2018, TPAMI 2019 · 1st place, ILSVRC 2017 classification |
| **Links** | [arXiv:1709.01507](https://arxiv.org/abs/1709.01507) · [PDF](https://arxiv.org/pdf/1709.01507) · [official code (Caffe)](https://github.com/hujie-frank/SENet) |
| **Problem** | CNN channels are treated as equally important everywhere; can the network learn *which* channels matter for *this* image? |
| **Key idea (one sentence)** | Squeeze each channel to one number (global average pool), pass the vector through a tiny 2-layer MLP with a sigmoid ("excitation"), and multiply every channel by its weight. |
| **Main claim** | Adding SE blocks to ResNet-50 costs ≈ 10 % more parameters (§3.3) and < 1 % more compute, and lowers ImageNet top-1 error from 24.80 % to 23.29 % (Table 2). |
| **Can I implement it?** | Yes: one small module (Fig. 1, Fig. 3) inside a ResNet block, which we already have. |

## Step 2 · Pass 2 notes: the method in my own words (§3, Fig. 3)

* **Squeeze** (Eq. 2): $z_c = \frac{1}{HW}\sum_{i,j} u_c(i, j)$, global average pooling: (B, C, H, W) → (B, C).
* **Excitation** (Eq. 3): $s = \sigma(W_2\,\delta(W_1 z))$ with $W_1 \in \mathbb{R}^{\frac{C}{r}\times C}$, $W_2 \in \mathbb{R}^{C\times\frac{C}{r}}$, δ = ReLU, σ = sigmoid, reduction ratio **r = 16**.
* **Scale** (Eq. 4): $\tilde{x}_c = s_c \cdot u_c$.
* In SE-ResNet (Fig. 3) the SE block sits on the **residual branch, before the addition** with the shortcut.
* *Pass-3 detail (§6):* on ResNets the authors **remove the biases** of the two FC layers. Details like this hide in the
  ablation section; we keep the biases (the simpler version of §3) and note the difference in the report.

**Shape table**

| tensor | shape |
|---|---|
| input u | (B, C, H, W) |
| after squeeze | (B, C) |
| after FC1 + ReLU | (B, C/r) |
| after FC2 + sigmoid | (B, C) → view (B, C, 1, 1) |
| output | (B, C, H, W), same as input |

## Step 3 · Equations → code
| Paper | PyTorch |
|---|---|
| Eq. 2, global average pooling | `u.mean(dim=(2, 3))` |
| Eq. 3, $W_1$, δ, $W_2$, σ | `nn.Linear(C, C // r)`, `torch.relu`, `nn.Linear(C // r, C)`, `torch.sigmoid` |
| Eq. 4, channel-wise scaling | `u * s.view(B, C, 1, 1)` (broadcasting) |

## Step 4 · Build bottom-up and test every block

### 📄 ↔ 💻 Paper ↔ Code: Figure 1 and Eq. (2)–(4) → `SEBlock`

<table>
<tr><th width="50%" align="left">📄 Paper · Hu et al. 2018, §3.1–3.2, Figure 1, Eq. (2)–(4)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_s_fig1.png" width="100%" alt="pc_s_fig1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_s_eq2.png" width="100%" alt="pc_s_eq2.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_s_eq3.png" width="100%" alt="pc_s_eq3.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_s_eq4.png" width="100%" alt="pc_s_eq4.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre>class SEBlock(nn.Module):
    def __init__(self, C, r=16):
        super().__init__()
        self.fc1 = nn.Linear(C, C // r)           # W1
        self.fc2 = nn.Linear(C // r, C)           # W2
    def forward(self, u):                         # u (B, C, H, W)
        B, C, _, _ = u.shape
        z = u.mean(dim=(2, 3))                                  # Eq. 2  F_sq
        s = torch.sigmoid(self.fc2(torch.relu(self.fc1(z))))    # Eq. 3  F_ex
        return u * s.view(B, C, 1, 1)                           # Eq. 4  F_scale</pre>
<b>Shapes:</b> (B, C, H, W) → squeeze (B, C) → FC1 + ReLU (B, C/r) → FC2 + sigmoid (B, C) → scale (B, C, H, W)
</td></tr>
</table>

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation (Hu et al., CVPR 2018), Eq. 2-4."""
    def __init__(self, C, r=16):
        super().__init__()
        self.fc1 = nn.Linear(C, C // r)
        self.fc2 = nn.Linear(C // r, C)

    def forward(self, u):
        B, C, _, _ = u.shape
        z = u.mean(dim=(2, 3))                              # squeeze   (B, C)
        s = torch.sigmoid(self.fc2(torch.relu(self.fc1(z))))  # excitation (B, C)
        return u * s.view(B, C, 1, 1)                       # scale

# Test 1: shape is unchanged
se = SEBlock(64)
u = torch.randn(2, 64, 8, 8)
assert se(u).shape == u.shape
# Test 2: a value I can check by hand: if fc2's weights and bias are 0, s = sigmoid(0) = 0.5 everywhere
with torch.no_grad():
    se.fc2.weight.zero_(); se.fc2.bias.zero_()
assert torch.allclose(se(u), 0.5 * u)
# Test 3: parameter count by hand: 64·4 + 4 + 4·64 + 64
assert C.count_params(SEBlock(64)) == 64 * 4 + 4 + 4 * 64 + 64
print("✅ SEBlock: shape, hand-computed value and parameter count all correct")

Now put it into the ResNet-50 bottleneck exactly where Figure 3 puts it, and compare the total parameter count with the
paper (§3.3: *"SE-ResNet-50 introduces ∼2.5 million additional parameters beyond the ∼25 million parameters required by
ResNet-50, corresponding to a ∼10% increase"*).

### 📄 ↔ 💻 Paper ↔ Code: Figure 3: where the SE block goes in a ResNet → `SEBottleneck`

<table>
<tr><th width="50%" align="left">📄 Paper · Hu et al. 2018, §4, Figures 2–3</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_s_fig23.png" width="100%" alt="pc_s_fig23.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre>def forward(self, x):
    identity = x if self.downsample is None else self.downsample(x)
    out = torch.relu(self.bn1(self.conv1(x)))      # Residual branch of Fig. 3:
    out = torch.relu(self.bn2(self.conv2(out)))    #   1x1, 3x3, 1x1 convolutions
    out = self.se(self.bn3(self.conv3(out)))       # SE on the branch, BEFORE (+)
    return torch.relu(out + identity)              # (+) with the shortcut, relu</pre>
</td></tr>
</table>

In [ ]:
class SEBottleneck(nn.Module):
    expansion = 4

    def __init__(self, cin, width, stride=1, r=16):
        super().__init__()
        cout = width * 4
        self.conv1 = nn.Conv2d(cin, width, 1, bias=False); self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, 3, stride, 1, bias=False); self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.se = SEBlock(cout, r)                                   # <- the only new line (Fig. 3)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = torch.relu(self.bn1(self.conv1(x)))
        out = torch.relu(self.bn2(self.conv2(out)))
        out = self.se(self.bn3(self.conv3(out)))                     # SE before the addition
        return torch.relu(out + identity)

def resnet_like(block, layers, num_classes=1000):
    """ResNet skeleton from Lab 7E (stem, 4 stages, pool, fc) for any block type."""
    mods, cin = [nn.Conv2d(3, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1)], 64
    for width, n, stride in zip([64, 128, 256, 512], layers, [1, 2, 2, 2]):
        for i in range(n):
            mods.append(block(cin, width, stride if i == 0 else 1)); cin = width * block.expansion
    return nn.Sequential(*mods, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(cin, num_classes))

import torchvision
n_se, n_plain = C.count_params(resnet_like(SEBottleneck, [3, 4, 6, 3])), C.count_params(torchvision.models.resnet50())
print(f"ResNet-50 (torchvision): {n_plain / 1e6:.2f} M · our SE-ResNet-50: {n_se / 1e6:.2f} M · added {(n_se - n_plain) / 1e6:.2f} M (paper: ~25 M + ~2.5 M)")
print(f"SE adds {100 * (n_se - n_plain) / n_plain:.1f} % parameters (paper: ≈ 10 %)")

## Step 5 · Overfit one batch

Before any real training: can a small SE network memorise 32 images? If not, there is a bug in the plumbing.

In [ ]:
Xtr, ytr, Xte, yte = C.load_digits_tensors()

class SmallBlock(nn.Module):
    def __init__(self, c, use_se):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, 1, 1, bias=False); self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(c)
        self.se = SEBlock(c, r=4) if use_se else nn.Identity()

    def forward(self, x):
        out = self.bn2(self.conv2(torch.relu(self.bn1(self.conv1(x)))))
        return torch.relu(self.se(out) + x)

def small_net(use_se, n_blocks=3, c=16):
    return nn.Sequential(nn.Conv2d(1, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
                         *[SmallBlock(c, use_se) for _ in range(n_blocks)],
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

C.overfit_one_batch(small_net(True), Xtr[:32], ytr[:32], F.cross_entropy, steps=150, lr=3e-3);

## Step 6 · A small experiment that tests the main claim (with error bars)

Claim: SE improves accuracy at almost no cost. Our miniature test: ResNet vs SE-ResNet on 8 × 8 digits, with only
**10 % of the training data** (so the task is not too easy), each trained with **3 different seeds** (one run proves
nothing: differences smaller than the seed-to-seed spread are noise).

In [ ]:
def run(use_se, seed, epochs=30, frac=0.1):
    C.seed_everything(seed)
    n = int(len(Xtr) * frac)
    m = small_net(use_se).to(device)
    opt = torch.optim.Adam(m.parameters(), 3e-3)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n)
        for i in range(0, n, 32):
            idx = perm[i:i + 32]
            loss = F.cross_entropy(m(Xtr[idx].to(device)), ytr[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return (m(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item(), C.count_params(m)

rows = {}
for name, use_se in [("ResNet (small)", False), ("SE-ResNet (small)", True)]:
    accs = [run(use_se, s)[0] for s in range(3)]
    rows[name] = (np.mean(accs), np.std(accs), run(use_se, 0, epochs=0)[1])
print(f"{'model':20s}{'params':>9s}{'test acc (3 seeds)':>22s}")
for k, (m, s, p) in rows.items():
    print(f"{k:20s}{p:9,d}{m:16.1%} ± {s:.1%}")

## Step 7 · Reproduction report (example)

* **What matched:** the module passes hand-made tests; SE-ResNet-50 adds ≈ 2.5 M parameters (+10 %) to ResNet-50, as §3.3 says.
* **Experiment:** on a tiny digits task with 10 % of the data, compare the mean ± std above. *Write what you see:* if the
  difference is smaller than the spread, say honestly that our small test **cannot confirm** the claim, and why
  (8 × 8 images have few channels to re-weight; the paper used ImageNet with 1.28 M images and 100+ epochs).
* **Differences from the paper:** r = 4 instead of 16 (we have only 16 channels), FC biases kept, Adam instead of SGD, tiny data.
* **With more compute I would:** train SE-ResNet-18 vs ResNet-18 on CIFAR-10 for 50 epochs on a Colab GPU, 3 seeds each,
  and try r ∈ {4, 8, 16} (the paper's Table 10 ablation).
* **AI use:** I asked an assistant to explain Eq. 3 and to review my shape table; all code written and tested by me.

---
# Part B · Your paper (copy this part for every paper)

## Step 1 · Paper card
| | |
|---|---|
| **Title** | |
| **Authors, venue, year** | |
| **Links** | arXiv · PDF · official code (if any) |
| **Problem** | |
| **Key idea (one sentence)** | |
| **Main claim (with the table/figure number)** | |
| **Why I chose it / can I implement it on Colab?** | |

## Step 2 · Pass 2 notes (my own words, not copied) + shape table
| tensor | shape |
|---|---|
| | |

## Step 3 · Figure → modules, equations → code
| Paper (figure / equation / table) | Module or line of PyTorch |
|---|---|
| | |

In [ ]:
# Step 4 · Build bottom-up. One class per box in the paper's figure; test each with asserts.

class MyBlock(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x

x = torch.randn(2, 3, 32, 32)
assert MyBlock()(x).shape == x.shape          # replace with the shapes from YOUR shape table
# C.trace_shapes(MyModel(), x)                # compare with the paper's figure
# print(C.count_params(MyModel()))            # compare with the paper's table, if it gives parameters

In [ ]:
# Step 5 · Overfit one batch
# C.overfit_one_batch(MyModel(), x_batch, y_batch, loss_fn)

In [ ]:
# Step 6 · Small experiment + ablation (at least 2 settings, ideally 3 seeds each), results table + plot

## Step 7 · Reproduction report
* **What matched:**
* **What did not match, and my best explanation:**
* **Differences from the paper (data, size, training):**
* **With more compute I would:**
* **AI use declaration** (what you asked, what you kept, what you checked yourself):
* **References** (paper, code you looked at, tutorials):